In [0]:
from pyspark.sql import functions as f  
from pyspark.sql import types as t      
import re                              
from bs4 import BeautifulSoup           
from urllib.parse import urljoin       
from datetime import datetime   
import logging        
from config import ROUTES, PipelineConfig  


## CVM - Informações Diárias 

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# config
BASE_URL  = "https://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/"
RAW_PATH  = f"{ROUTES.RAW_PATH}/cvm_informe_diario/"
PATTERN   = re.compile(r"inf_diario_fi_(\d{6})\.zip")

NOME_TABELA  = f"bronze_cvm_informe_diario" 
BRONZE_PATH  = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

### 1. Verificando os arquivos

Essa logica prioriza dos de M-0 e M-1, devido a atualização na base origem. Casos os dados forem de meses M-2 + serão atualizado aos domingos quando a base origem atualiza. Importante logica para controle de carga da pipeline e para FinOps.

In [0]:
def regra_finops(response, pattern:re.Pattern) -> list:
    # ========================================== 
    #           REGRA DE ATUALIZAÇÃO
    #===========================================
    # M-0 e M-1 > atualizam diariamente
    # M-2 + > Atualizam semanalmente nos domingos 

    soup = BeautifulSoup(response, "html.parser")

    hoje = datetime.today()
    current_month = hoje.month
    current_year = hoje.year

    dia_da_semana_update = hoje.weekday() == 6 # Domingo

    files = []

    for link in soup.find_all("a", href=True):
        href = link["href"]
        match = pattern.match(href)

        if match:
            file_yyyymm = match.group(1)
            file_year = int(file_yyyymm[:4])
            file_month = int(file_yyyymm[4:6])
            
            diff_month = (current_year - file_year) * 12 + (current_month - file_month)
            
            if diff_month <= 1:
                files.append(urljoin(BASE_URL, href))

            elif diff_month >= 2 and dia_da_semana_update:
                files.append(urljoin(BASE_URL, href))

    return files


In [0]:
response = PipelineConfig.retorno_text_url(url=BASE_URL)

files = regra_finops(response=response, pattern=PATTERN)

log.info(f"Arquivos para processar: {files}")   

### 2. Extraindo os arquivos depositando na camada raw

In [0]:
for zip_url in files:
    log.info(f"Processando: {zip_url}")
    PipelineConfig.baixar_e_extrair_zip(url=zip_url, raw_path=RAW_PATH)


### 3. Salvar em camada Bronze Particionada

Nesta etapa, realizamos a ***normalização*** das colunas. Nos arquivos recentes, foi incluída a coluna **ID_SUBCLASSE**; já nos registros de anos anteriores, essa coluna é inexistente e a nomenclatura das demais difere do padrão atual. Esse processo garante a padronização e a ordenação ***correta*** dos dados para o consumo.

In [0]:
# 1. Lista todos os arquivos CSV dentro do volume
arquivos_raw = [file.path for file in dbutils.fs.ls(RAW_PATH) if file.name.endswith('.csv')]

SCHEMA_NOVO = t.StructType([
    t.StructField("TP_FUNDO_CLASSE",   t.StringType(), True),
    t.StructField("CNPJ_FUNDO_CLASSE", t.StringType(), True),
    t.StructField("ID_SUBCLASSE",      t.StringType(), True),
    t.StructField("DT_COMPTC",         t.StringType(), True),
    t.StructField("VL_TOTAL",          t.StringType(), True),
    t.StructField("VL_QUOTA",          t.StringType(), True),
    t.StructField("VL_PATRIM_LIQ",     t.StringType(), True),
    t.StructField("CAPTC_DIA",         t.StringType(), True),
    t.StructField("RESG_DIA",          t.StringType(), True),
    t.StructField("NR_COTST",          t.StringType(), True),
])

SCHEMA_ANTIGO = t.StructType([
    t.StructField("TP_FUNDO",      t.StringType(), True),
    t.StructField("CNPJ_FUNDO",    t.StringType(), True),
    t.StructField("DT_COMPTC",     t.StringType(), True),
    t.StructField("VL_TOTAL",      t.StringType(), True),
    t.StructField("VL_QUOTA",      t.StringType(), True),
    t.StructField("VL_PATRIM_LIQ", t.StringType(), True),
    t.StructField("CAPTC_DIA",     t.StringType(), True),
    t.StructField("RESG_DIA",      t.StringType(), True),
    t.StructField("NR_COTST",      t.StringType(), True),
])

# Listas separadas para os arquivos novos e legados
arquivos_antigo = list()
arquivos_novo = list()
dfs_para_unir = list()


# 2. Lê e separar os arquivos
for path in arquivos_raw:
    colunas = spark.read.option("sep", ";").option("header", "true").csv(path).columns
    if "TP_FUNDO" in colunas:
        arquivos_antigo.append(path)
    else:
        arquivos_novo.append(path)


# 3. Lê arquivos Com a padronização correta
if arquivos_novo:
    df_novo = (spark.read
            .schema(SCHEMA_NOVO)
            .option("sep", ";")
            .option("header", "true")
            .csv(arquivos_novo)
            )
    dfs_para_unir.append(df_novo)

# 4. Lê arquivos Com a padronização errada

if arquivos_antigo:
    df_antigo = (spark.read
        .schema(SCHEMA_ANTIGO)
        .option("sep", ";")
        .option("header", "true")
        .csv(arquivos_antigo)
        .withColumnRenamed("TP_FUNDO", "TP_FUNDO_CLASSE")
        .withColumnRenamed("CNPJ_FUNDO", "CNPJ_FUNDO_CLASSE")
        .withColumn("ID_SUBCLASSE", f.lit(None).cast("string")))
    dfs_para_unir.append(df_antigo)


# 5. 1 único union entre 2 DataFrames — plano flat
# Valida se há dados para processar antes de fazer a união
if len(dfs_para_unir) == 2:
    df_cvm = dfs_para_unir[0].unionByName(dfs_para_unir[1], allowMissingColumns=True)
elif len(dfs_para_unir) == 1:
    df_cvm = dfs_para_unir[0]
else:
    print("Nenhum arquivo CSV encontrado para processar.")
    dbutils.notebook.exit("Sucesso: Sem arquivos novos")

# 6. Criação de metadados
df_cvm = (df_cvm
      .withColumn("_source_url", f.lit(BASE_URL))
      .withColumn("_ingest_timestamp", f.current_timestamp())
      .withColumn("data_processamento", f.lit(DATA_PROC))
)


# 7. Escrita na Bronze 
(df_cvm.write 
    .mode("overwrite") 
    .option("replaceWhere", f"data_processamento = {DATA_PROC}") 
    .option("mergeSchema", "true")
    .partitionBy("data_processamento") 
    .format("delta") 
    .saveAsTable(BRONZE_PATH)
)
